# 01 — CRCP Benthic Cover Data Exploration

This notebook profiles the NOAA CRCP benthic cover dataset for Hawaii,
accessed from ERDDAP (`CRCP_Benthic_Cover_Hawaii`). It documents the
schema, taxonomy hierarchy, data distributions, and quality issues that
inform the design of the processing pipeline.

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
from collections import Counter, defaultdict
from erddapy import ERDDAP
from src.config import get_source

## 1. Connect to ERDDAP and fetch data

In [ ]:
cfg = get_source("hawaii")
print(f"Dataset: {cfg.dataset_id}")
print(f"Server:  {cfg.erddap_server}")

e = ERDDAP(
    server=cfg.erddap_server,
    protocol="tabledap",
)
e.dataset_id = cfg.dataset_id
e.variables = [
    "latitude", "longitude", "region_name", "island", "site",
    "reef_zone", "depth_bin", "min_depth", "max_depth",
    "image_name", "image_url", "obs_year", "date_",
    "analyst", "tier_1", "category_name",
    "tier_2", "subcategory_name", "tier_3", "genera_name",
]

print("Fetching data from ERDDAP (this may take a minute)...")
df = e.to_pandas()
print(f"Rows fetched: {len(df):,}")
df.head()

## 2. Schema overview

In [ ]:
print(f"Shape: {df.shape}")
print(f"\nColumns and dtypes:")
for col in df.columns:
    non_null = df[col].notna().sum()
    print(f"  {col:30s}  {str(df[col].dtype):10s}  ({non_null:,} non-null)")

## 3. Distributions

In [ ]:
print("=== Survey Years ===")
print(df["obs_year"].value_counts().sort_index().to_string())

print(f"\n=== Islands ({df['island'].nunique()}) ===")
print(df["island"].value_counts().sort_index().to_string())

print(f"\n=== Reef Zones ===")
print(df["reef_zone"].value_counts().to_string())

print(f"\n=== Depth Bins ===")
print(df["depth_bin"].value_counts().to_string())

print(f"\nUnique sites:  {df['site'].nunique():,}")
print(f"Unique images: {df['image_name'].nunique():,}")
print(f"Unique coords: {df.groupby(['latitude','longitude']).ngroups:,}")

## 4. Tier taxonomy tree

In [ ]:
tier_map = defaultdict(lambda: defaultdict(set))
for _, row in df[["tier_1", "category_name", "tier_2", "subcategory_name", "tier_3", "genera_name"]].drop_duplicates().iterrows():
    t1 = f"{row['tier_1']} ({row['category_name']})"
    t2 = f"{row['tier_2']} ({row['subcategory_name']})"
    t3 = f"{row['tier_3']} ({row['genera_name']})" if pd.notna(row['tier_3']) and row['tier_3'] != '' else None
    if t3:
        tier_map[t1][t2].add(t3)
    else:
        tier_map[t1][t2]  # ensure key exists

for t1 in sorted(tier_map):
    print(f"\n{t1}")
    for t2 in sorted(tier_map[t1]):
        children = sorted(tier_map[t1][t2])
        print(f"  └─ {t2}")
        for t3 in children:
            print(f"       └─ {t3}")

## 5. Points per image

In [ ]:
pts_per_img = df.groupby("image_name").size()
print(f"Points per image — min: {pts_per_img.min()}, max: {pts_per_img.max()}, "
      f"median: {pts_per_img.median()}, mean: {pts_per_img.mean():.1f}")
print(f"\nDistribution:")
print(pts_per_img.value_counts().sort_index().to_string())

## 6. Multi-year site coverage

In [ ]:
site_years = df.groupby("site")["obs_year"].apply(lambda s: tuple(sorted(s.unique())))
multi = site_years[site_years.apply(len) > 1]
print(f"Sites surveyed in 1 year only: {(site_years.apply(len) == 1).sum()}")
print(f"Sites surveyed in 2+ years:    {len(multi)}")
if len(multi) > 0:
    print("\nYear combinations:")
    print(multi.value_counts().to_string())

## 7. Tape/Wand and Unclassified prevalence

In [ ]:
tier1_counts = df["tier_1"].value_counts()
total = len(df)
for label in ["TW", "UC"]:
    n = tier1_counts.get(label, 0)
    print(f"{label}: {n:,} points ({100*n/total:.2f}%)")

tw_per_img = df[df["tier_1"] == "TW"].groupby("image_name").size()
print(f"\nImages containing TW points: {len(tw_per_img):,} / {df['image_name'].nunique():,}")
if len(tw_per_img) > 0:
    print(f"TW points per affected image — min: {tw_per_img.min()}, max: {tw_per_img.max()}, median: {tw_per_img.median()}")

## 8. Coordinate quality check

In [ ]:
null_lat = df["latitude"].isna().sum()
null_lon = df["longitude"].isna().sum()
print(f"Null latitude:  {null_lat}")
print(f"Null longitude: {null_lon}")

print(f"\nLatitude  range: {df['latitude'].min():.6f} to {df['latitude'].max():.6f}")
print(f"Longitude range: {df['longitude'].min():.6f} to {df['longitude'].max():.6f}")

# Check for sites with multiple coordinate sets
site_coords = df.groupby("site").agg(
    n_coords=("latitude", lambda s: len(set(zip(s, df.loc[s.index, "longitude"]))))
)
multi_coord = site_coords[site_coords["n_coords"] > 1]
print(f"\nSites with single coordinate: {(site_coords['n_coords'] == 1).sum()}")
print(f"Sites with multiple coordinates: {len(multi_coord)}")

## Summary

Key findings for pipeline design:

1. **~470k point-level rows** across ~47k images and ~1,550 sites
2. **10 points per image** (standard CRCP stratified random protocol)
3. **3-tier taxonomy** built into the data — no external label mapping needed
4. **TW (Tape/Wand)** points (~2.3%) should be excluded and covers renormalized
5. **UC (Unclassified)** points (~1.9%) kept as "Other"
6. **No individual sites** span multiple survey years → temporal trends at island level
7. **Image URLs** available for every point → representative thumbnails in popups
8. **Coordinates** are clean with only minor multi-coordinate sites